## Классификация текстов с использованием предобученных языковых моделей.

В данном задании вам предстоит обратиться к задаче классификации текстов и решить ее с использованием предобученной модели BERT.

In [62]:
import json
# do not change the code in the block below
# __________start of block__________
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

%matplotlib inline
# __________end of block__________

Обратимся к набору данных SST-2. Holdout часть данных (которая понадобится вам для посылки) доступна по ссылке ниже.

In [63]:
# do not change the code in the block below
# __________start of block__________

!wget https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
# __________end of block__________

--2026-11-01 23:35:58--  https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51581 (50K) [text/plain]
Saving to: ‘texts_holdout.json.1’

texts_holdout.json. 100%[===================>]  50.37K   268KB/s    in 0.2s    

2026-11-01 23:35:59 (268 KB/s) - ‘texts_holdout.json.1’ saved [51581/51581]



In [3]:
# do not change the code in the block below
# __________start of block__________
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]
with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)
# __________end of block__________

Весь остальной код предстоит написать вам.

Для успешной сдачи на максимальный балл необходимо добиться хотя бы __84.5% accuracy на тестовой части выборки__.

In [64]:
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

# По умолчанию классификатором здесь будет один слой
# При желании можно сделать классификатором MLP с 2 и более слоями
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-large-cased")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-large-cased",
    num_labels=2
)

# Так как нам нужно работать с прямоугольными тензорами мы строим тензор 
# с количеством столбцов равным максимальной длине последовательностей
# за это отвечает padding = True, если в последовательности токенов меньше
# он заполняет остатки специальными токенами [PAD]
# А настоящие токены от [PAD] он отличает с помощью attention_mask
# Токены:
# [CLS] I love cats [SEP] [PAD] [PAD]
# Маска:
#  1    1   1    1     1     0     0
# truncation=True обрезает тексты до max_length

# Токенизатор возвращает не один тензор, а словарь с несколькими тензорами.
# В случае с BERT получим словарь со следующими ключами:
# dict_keys([
#    'input_ids',
#    'token_type_ids',
#    'attention_mask'
# ])
# input_ids это индексы наших токенов
# token_type_ids это индексы сегментов

train_inputs = tokenizer(
    list(texts_train),
    padding=False, 
    max_length=256,
    truncation=True,
)

test_inputs = tokenizer(
    list(texts_test),
    padding=False,
    max_length=256,
    truncation=True,
)

holdout_inputs = tokenizer(
    list(texts_holdout),
    padding=False,
    max_length=256,
    truncation=True
)

train_labels = torch.tensor(y_train, dtype=torch.long)
test_labels = torch.tensor(y_test, dtype=torch.long)
holdout_labels = torch.zeros(len(texts_holdout), dtype=torch.long)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

class BertDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: value[idx]
            for key, value in self.inputs.items()
        }
        item["labels"] = int(self.labels[idx])
        return item

train_dataset = BertDataset(train_inputs, train_labels)
test_dataset = BertDataset(test_inputs, test_labels)
holdout_dataset = BertDataset(holdout_inputs, holdout_labels)

train_loader = DataLoader(train_dataset, batch_size=8, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=8, collate_fn=collator)
holdout_loader = DataLoader(
    holdout_dataset,
    batch_size=8,
    collate_fn=collator
)

for param in model.bert.parameters():
    param.requires_grad = False

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3402.62it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-large-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

In [65]:
def extract_embeddings(model, loader):
    model.bert.eval()
    embeddings = []

    with torch.no_grad():
        for batch in loader:
            outputs = model.bert(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch["token_type_ids"]
            )

            embeddings.append(outputs.pooler_output)

    return torch.cat(embeddings, dim=0)

X_train = extract_embeddings(model, train_loader)
X_test = extract_embeddings(model, test_loader)
X_holdout = extract_embeddings(model, holdout_loader)
y_train = train_labels
y_test = test_labels
y_holdout = holdout_labels

train_dataset = TensorDataset(X_train, y_train)

train_loader_classifier = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [66]:
def train(model, loss_function, optim, n_epochs, f):
    for epoch in range(n_epochs+1):
        total_loss = 0
        for X_batch, y_batch in train_loader_classifier:
            optim.zero_grad()

            logits = model(X_batch)
            loss = loss_function(logits, y_batch)
            loss.backward()
            optim.step()

            total_loss += loss.item()

        if epoch % 100 == 0:
            print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_loader_classifier):.4f}")

    torch.save(model.state_dict(), f)

def accuracy(model, X, y):
    model.eval()

    with torch.no_grad():
        logits = model(X)
        predictions = logits.argmax(dim=1)

        return (predictions == y).float().mean().item()

In [67]:
logreg = nn.Linear(1024, 2)
loss_function = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(logreg.parameters(), lr=1e-3)

train(logreg, loss_function, optim, 500, "logreg.pth")

train_acc = accuracy(logreg, X_train, y_train)
test_acc = accuracy(logreg, X_test, y_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")

Epoch 1: loss = 0.7014
Epoch 101: loss = 0.3499
Epoch 201: loss = 0.3373
Epoch 301: loss = 0.3244
Epoch 401: loss = 0.3257
Epoch 501: loss = 0.3157
Train Accuracy: 0.8788
Test Accuracy:  0.8458


In [68]:
mlp = nn.Sequential(
    nn.Linear(1024, 512),
    nn.ReLU(),
    #nn.Dropout(0.1), 
    nn.Linear(512, 2)
)

loss_function = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(mlp.parameters(), lr=2e-4)

train(mlp, loss_function, optim, 300, "mlp.pth")

train_acc = accuracy(mlp, X_train, y_train)
test_acc = accuracy(mlp, X_test, y_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")

Epoch 1: loss = 0.6943
Epoch 101: loss = 0.3548
Epoch 201: loss = 0.3238
Epoch 301: loss = 0.3077
Train Accuracy: 0.8832
Test Accuracy:  0.8505


#### Сдача взадания в контест
Сохраните в словарь `out_dict` вероятности принадлежности к первому (положительному) классу

In [69]:
mlp.eval()

with torch.no_grad():
    out_dict = {
        "train": torch.softmax(mlp(X_train), dim=1)[:, 1].tolist(),
        "test": torch.softmax(mlp(X_test), dim=1)[:, 1].tolist(),
        "holdout": torch.softmax(mlp(X_holdout), dim=1)[:, 1].tolist()
    }

Несколько `assert`'ов для проверки вашей посылки:

In [71]:
assert isinstance(out_dict["train"], list), "Object must be a list of floats"
assert isinstance(out_dict["train"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["train"]) == 5000
), "The predicted probas list length does not match the train set size"

assert isinstance(out_dict["test"], list), "Object must be a list of floats"
assert isinstance(out_dict["test"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["test"]) == 1920
), "The predicted probas list length does not match the test set size"

assert isinstance(out_dict["holdout"], list), "Object must be a list of floats"
assert isinstance(out_dict["holdout"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["holdout"]) == 500
), "The predicted probas list length does not match the holdout set size"

Запустите код ниже для генерации посылки.

In [72]:
# do not change the code in the block below
# __________start of block__________
FILENAME = "submission_dict_hw_text_classification_with_bert.json"

with open(FILENAME, "w") as iofile:
    json.dump(out_dict, iofile)
print(f"File saved to `{FILENAME}`")
# __________end of block__________

File saved to `submission_dict_hw_text_classification_with_bert.json`


На этом задание завершено. Поздравляем!